# 06. HiCache、Disaggregation 与新特性落点

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## HiCache 是 RadixAttention 的分层扩展

RadixAttention 把可复用 KV 留在 GPU 空闲空间中。HiCache 把这个想法扩展为：

- L1：GPU KV cache。
- L2：本机 host memory KV cache。
- L3：跨实例共享的外部存储，如 Mooncake、HF3FS、NIXL、AIBrix、file backend。

关键变化是 metadata tree 需要同时知道 device/host/storage 的命中情况，prefill 需要把 host/storage 命中的 KV load back 到 GPU。


In [ ]:
for path in [
    "docs/advanced_features/hicache_design.md",
    "python/sglang/srt/mem_cache/hiradix_cache.py",
    "python/sglang/srt/managers/cache_controller.py",
    "python/sglang/srt/mem_cache/hicache_storage.py",
]:
    print("\n==", path)
    print((ROOT / path).exists())


## HiRadixCache 的关键状态

`HiRadixCache` 继承 `RadixCache`，但节点上除了 device value，还可能有 host value、write-through pending id、hash values。它通过 `HiCacheController` 处理 GPU<->CPU 和 CPU<->L3 的传输。


In [ ]:
show_lines("python/sglang/srt/mem_cache/hiradix_cache.py", 72, 185)


### `HiRadixCache.__init__`：在 RadixCache 外面加 L2/L3 控制器

```python
# python/sglang/srt/mem_cache/hiradix_cache.py:72-185
  72: class HiRadixCache(RadixCache):
  73: 
  74:     def __init__(self, params: CacheInitParams, server_args: ServerArgs):
  75:         self._enable_metrics_flag = params.enable_metrics
      # 注：HiCache 是否采集 metrics 的总开关，来自 cache init params。
  76: 
  77:         self.page_size = params.page_size
      # 注：HiCache 所有 L1/L2/L3 KV 操作都按 page 对齐，page_size 是 load-back/prefetch 的基本粒度。
  78:         self.kv_cache = params.token_to_kv_pool_allocator.get_kvcache()
      # 注：当前 GPU device KV pool；HiRadixCache 根据它的具体类型选择 host pool 布局。
  79: 
  80:         if isinstance(self.kv_cache, MHATokenToKVPool):
      # 注：普通 MHA KV cache 使用与 device pool 匹配的 host pool，支持 L2 写回/加载。
  81:             self.token_to_kv_pool_host = get_mha_host_pool_cls(self.kv_cache)(
      # 注：HiCache L2 host KV pool，保存从 device 写回或从 L3 预取的 KV page。
  82:                 self.kv_cache,
  83:                 server_args.hicache_ratio,
  84:                 server_args.hicache_size,
  85:                 self.page_size,
  86:                 server_args.hicache_mem_layout,
  87:                 allocator_type=server_args.hicache_storage_backend,
  88:             )
  89:         elif isinstance(self.kv_cache, DSATokenToKVPool):
      # 注：DSA cache 需要等解析 storage extra_config 后走 hybrid attach，不能在这里直接创建 host pool。
  90:             # Filled by attach_hybrid_dsa_pool_to_hiradix_cache after storage extra_config is parsed.
  91:             self.token_to_kv_pool_host = None
  92:         elif isinstance(self.kv_cache, MLATokenToKVPool):
      # 注：MLA KV cache 使用专门 host pool，因为 MLA 的 KV layout 与普通 MHA 不同。
  93:             self.token_to_kv_pool_host = MLATokenToKVPoolHost(
  94:                 self.kv_cache,
  95:                 server_args.hicache_ratio,
  96:                 server_args.hicache_size,
  97:                 self.page_size,
  98:                 server_args.hicache_mem_layout,
  99:                 allocator_type=server_args.hicache_storage_backend,
 100:             )
 101:         else:
 102:             raise ValueError("HiRadixCache only supports MHA, MLA, and DSA models")
 103: 
 104:         self.tp_group = params.tp_cache_group
      # 注：HiCache 传输和命中查询需要与 TP cache group 对齐，跨 rank 时按这个 group 同步。
 105:         self.attn_cp_group = params.attn_cp_cache_group
      # 注：attention context-parallel cache group，用于 HiCache load-back/prefetch 的跨 rank 同步。
 106:         self.attn_tp_group = params.attn_tp_cache_group
      # 注：attention tensor-parallel cache group，用于 HiCache storage 命中和预取一致性。
 107:         self.pp_group = params.pp_cache_group
      # 注：pipeline-parallel cache group，HiCache controller 需要它处理 PP 场景下的传输协调。
 108:         self.tp_world_size = torch.distributed.get_world_size(group=self.tp_group)
 109:         self.pp_rank = params.pp_rank
 110:         self.pp_size = params.pp_size
 111:         self.enable_storage = server_args.hicache_storage_backend is not None
      # 注：是否启用 L3 storage backend；未配置时 HiCache 只提供 L1/L2。
 112:         self.enable_storage_metrics = self.enable_storage and params.enable_metrics
      # 注：只有启用 L3 且 metrics 打开时才采集 storage 相关指标。
 113:         self.extra_metric_labels = server_args.extra_metric_labels
 114: 
 115:         (
 116:             extra_config,
 117:             prefetch_threshold,
 118:             prefetch_timeout_config,
 119:             hicache_storage_pass_prefix_keys,
 120:         ) = self._parse_storage_backend_extra_config(
      # 注：关键调用：`self._parse_storage_backend_extra_config` - 解析 L3 storage backend 的额外配置，得到预取阈值、超时策略和 prefix-key 传递策略。
 121:             server_args.hicache_storage_backend_extra_config
 122:         )
 123:         # TODO: support more timeout check functions
 124:         self.is_prefetch_timeout = self._prefetch_timeout_check_linear_func
      # 注：当前使用的 prefetch 超时判断函数，默认是线性超时策略。
 125:         self.prefetch_stop_policy = server_args.hicache_storage_prefetch_policy
      # 注：L3 预取停止策略，决定命中不足、超时或资源不足时如何撤销/截断。
 126: 
 127:         self.load_cache_event = threading.Event()
      # 注：load-back/prefetch 完成通知事件，controller 和 cache 主路径通过它协调异步传输。
 128:         if isinstance(self.kv_cache, DSATokenToKVPool):
 129:             attach_hybrid_dsa_pool_to_hiradix_cache(
      # 注：关键调用：`attach_hybrid_dsa_pool_to_hiradix_cache` - DSA KV layout 需要专门的 hybrid host pool/controller 装配，不能走普通 MHA/MLA 分支。
 130:                 self,
 131:                 params,
 132:                 server_args,
 133:                 extra_config=extra_config,
 134:                 prefetch_threshold=prefetch_threshold,
 135:                 enable_storage_metrics=self.enable_storage_metrics,
 136:                 load_cache_event=self.load_cache_event,
 137:                 attn_cp_group=self.attn_cp_group,
 138:                 attn_tp_group=self.attn_tp_group,
 139:             )
 140:         else:
 141:             self.cache_controller = HiCacheController(
      # 注：HiCache 异步传输控制器，负责 host pool 分配、device<->host copy、storage prefetch 和 ACK。
 142:                 params.token_to_kv_pool_allocator,
 143:                 self.token_to_kv_pool_host,
 144:                 self.page_size,
 145:                 self.tp_group,
 146:                 load_cache_event=self.load_cache_event,
 147:                 attn_cp_group=self.attn_cp_group,
 148:                 attn_tp_group=self.attn_tp_group,
 149:                 pp_group=self.pp_group,
 150:                 write_policy=server_args.hicache_write_policy,
 151:                 io_backend=server_args.hicache_io_backend,
 152:                 storage_backend=server_args.hicache_storage_backend,
 153:                 prefetch_threshold=prefetch_threshold,
 154:                 model_name=server_args.served_model_name,
 155:                 storage_backend_extra_config=extra_config,
 156:                 enable_storage_metrics=self.enable_storage_metrics,
 157:             )
 158:         self._apply_storage_runtime_config(
      # 注：关键调用：`self._apply_storage_runtime_config` - 把 storage backend、阈值、超时和 metrics 配置写入 HiRadixCache 运行时状态。
 159:             storage_backend=server_args.hicache_storage_backend,
 160:             prefetch_threshold=prefetch_threshold,
 161:             prefetch_timeout_config=prefetch_timeout_config,
 162:             hicache_storage_pass_prefix_keys=hicache_storage_pass_prefix_keys,
 163:             enable_storage=self.enable_storage,
 164:             enable_storage_metrics=self.enable_storage_metrics,
 165:             extra_metric_labels=self.extra_metric_labels,
 166:         )
 167: 
 168:         # record the nodes with ongoing write through
 169:         self.ongoing_write_through = {}
      # 注：记录正在写穿到 host/storage 的 radix 节点，避免节点 mutation 与异步写回冲突。
 170:         # record the node segments with ongoing load back
 171:         self.ongoing_load_back = {}
      # 注：记录正在从 host 加载回 device 的节点片段，避免重复 load-back 同一段 KV。
 172:         # record the ongoing prefetch requests
 173:         self.ongoing_prefetch = {}
      # 注：按 request id 记录正在进行的 L3->host 预取，terminate/progress 需要这份账本。
 174:         self.ongoing_backup = {}
      # 注：记录后台 backup 操作，防止驱逐或节点更新时丢失仍在传输的 KV。
 175:         # track per-request tokens loaded from storage (L3 hits)
 176:         # key: request_id, value: number of tokens actually loaded from storage
 177:         self.prefetch_loaded_tokens_by_reqid: dict[str, int] = {}
      # 注：按请求统计实际从 L3 加载的 token 数，用于 metrics 和后续调度判断。
 178:         self.work_list: List[torch.distributed.Work] = []
      # 注：保存 torch distributed 异步 work 句柄，确保跨 rank HiCache 通信完成后再回收状态。
 179:         # todo: dynamically adjust the threshold
 180:         self.write_through_threshold = (
      # 注：write-through 策略下更积极地写回 host/storage，write-back 策略则用更高阈值减少传输。
 181:             1 if server_args.hicache_write_policy == "write_through" else 2
 182:         )
 183:         self.load_back_threshold = 10
      # 注：load-back 最小收益阈值，命中过短时不值得把 host KV 搬回 device。
 184: 
 185:         # Detach storage backend automatically on process shutdown
```

**读这段时抓住：**

- 根据底层 KV pool 类型创建 host pool；MHA、MLA、DSA 的 host memory layout 不同。
- `enable_storage` 由 storage backend 是否存在决定；只开 L2 和开 L3 是两种不同模式。
- `HiCacheController` 接管 GPU<->CPU 和 CPU<->storage 的异步传输，Radix tree 仍负责 metadata。
- `ongoing_write_through`、`ongoing_load_back`、`ongoing_prefetch` 是防止异步 I/O 和 tree mutation 打架的关键账本。


## 三个动作：local match、prefetch、write-back

- `match_prefix`：先查本地 tree，返回 device hit + host hit。
- `load_back` / `init_load_back`：把 host hit 的 KV 搬回 device。
- `prefetch_from_storage` / `check_prefetch_progress`：从 L3 异步预取到 host。
- `write_backup` / `write_backup_storage`：把热 KV 或被驱逐 KV 写回 host/storage。


In [ ]:
for name in ["match_prefix", "load_back", "init_load_back", "prefetch_from_storage", "check_prefetch_progress", "write_backup", "write_backup_storage", "evict"]:
    rows = find_defs("python/sglang/srt/mem_cache/hiradix_cache.py", {name})
    print(name, rows)


In [ ]:
show_lines("python/sglang/srt/mem_cache/hiradix_cache.py", 1241, 1345)
show_lines("python/sglang/srt/mem_cache/hiradix_cache.py", 1438, 1530)


### `init_load_back`：L2 host hit 什么时候搬回 GPU

```python
# python/sglang/srt/mem_cache/hiradix_cache.py:1212-1265
1212:     def init_load_back(
1213:         self,
1214:         params: InitLoadBackParams,
1215:     ):
1216:         last_node = params.best_match_node
1217:         mem_quota = params.mem_quota
1218:         if last_node.evicted:
1219:             loading_values = self.load_back(last_node, mem_quota)
1220:             if loading_values is not None:
1221:                 logger.debug(
1222:                     f"loading back {len(loading_values)} tokens for node {last_node.id}"
1223:                 )
1224:                 return loading_values, last_node
1225: 
1226:             while last_node.evicted:
1227:                 last_node = last_node.parent
1228: 
1229:         return (
1230:             self._empty_match_result.device_indices,
1231:             last_node,
1232:         )
1233: 
1234:     def query_storage_hit_length(
1235:         self,
1236:         last_host_node: TreeNode,
1237:         new_input_tokens: List[int],
1238:         last_hash: Optional[str] = None,
1239:         prefix_keys: Optional[List[str]] = None,
1240:     ) -> int:
1241:         if not self.enable_storage or self.cache_controller.prefetch_rate_limited():
1242:             return 0
1243: 
1244:         prefetch_key = RadixKey(
1245:             new_input_tokens,
1246:             extra_key=last_host_node.key.extra_key,
1247:             is_bigram=self.is_eagle,
1248:         ).page_aligned(self.page_size)
1249:         if len(prefetch_key) < self.prefetch_threshold:
1250:             return 0
1251: 
1252:         operation = PrefetchOperation(
1253:             "__storage_hit_query__",
1254:             self.cache_controller.mem_pool_host.get_dummy_flat_data_page()[:0],
1255:             prefetch_key,
1256:             last_hash,
1257:             prefix_keys,
1258:         )
1259:         hash_values, storage_hit_count = self.cache_controller._storage_hit_query(
1260:             operation
1261:         )
1262:         storage_hit_count_tensor = torch.tensor(storage_hit_count, dtype=torch.int)
1263:         self._all_reduce_attn_groups(
1264:             storage_hit_count_tensor, torch.distributed.ReduceOp.MIN
1265:         )
```

**读这段时抓住：**

- `best_match_node` 是 load-back 的锚点，普通 RadixCache 不需要这个复杂度，HiCache 需要。
- 如果 L3 启用且 host hit 后还有足够长的 storage 可能命中，会触发 prefetch。
- load-back 和 prefetch 是两个层级：前者把 L2 -> L1，后者把 L3 -> L2。


### `prefetch_from_storage`：L3 预取先占 host page，再异步发起 I/O

```python
# python/sglang/srt/mem_cache/hiradix_cache.py:1471-1530
1471:     def prefetch_from_storage(
1472:         self,
1473:         req_id: str,
1474:         last_host_node: TreeNode,
1475:         new_input_tokens: List[int],
1476:         last_hash: Optional[str] = None,
1477:         prefix_keys: Optional[List[str]] = None,
1478:     ):
1479:         prefetch_key = RadixKey(
1480:             new_input_tokens,
1481:             extra_key=last_host_node.key.extra_key,
1482:             is_bigram=self.is_eagle,
1483:         )
1484:         # align the number of fetching tokens to the page size
1485:         prefetch_key = prefetch_key.page_aligned(self.page_size)
1486:         prefetch_length = len(prefetch_key)
1487:         if (
1488:             not self.enable_storage
1489:             or prefetch_length < self.prefetch_threshold
1490:             or self.cache_controller.prefetch_rate_limited()
1491:         ):
1492:             return
1493: 
1494:         last_host_node.protect_host()
1495:         host_indices = self.cache_controller.mem_pool_host.alloc(prefetch_length)
1496:         if host_indices is None:
1497:             self.evict_host(prefetch_length)
      # 注：关键调用：`self.evict_host` - 从 host KV pool 中驱逐页面，为新的 load-back/prefetch 腾空间。
1498:             host_indices = self.cache_controller.mem_pool_host.alloc(prefetch_length)
1499:         if host_indices is None:
1500:             available_size = self.cache_controller.mem_pool_host.available_size()
1501:             prefetch_length = available_size - (available_size % self.page_size)
1502:             if prefetch_length >= self.prefetch_threshold:
1503:                 prefetch_key = prefetch_key[:prefetch_length]
1504:                 host_indices = self.cache_controller.mem_pool_host.alloc(
1505:                     prefetch_length
1506:                 )
1507:                 if host_indices is None:
1508:                     last_host_node.release_host()
1509:                     return
1510:             else:
1511:                 last_host_node.release_host()
1512:                 # no sufficient host memory for prefetch
1513:                 return
1514:         operation = self.cache_controller.prefetch(
      # 注：`operation` 接收 `self.cache_controller.prefetch` 的结果：发起从 L3 storage 到 host pool 的异步预取。
1515:             req_id,
1516:             host_indices,
1517:             prefetch_key,
1518:             last_hash,
1519:             prefix_keys,
1520:             **self._get_extra_pools(),
1521:         )
1522:         self.ongoing_prefetch[req_id] = (
1523:             last_host_node,
1524:             prefetch_key,
1525:             host_indices,
1526:             operation,
1527:         )
1528:         self.cache_controller.prefetch_tokens_occupied += len(prefetch_key)
1529: 
1530:     def _insert_helper_host(
```

**读这段时抓住：**

- `prefetch_key` 会按 page 对齐，避免 L3 读写粒度和 host pool 粒度不一致。
- 如果 host pool 不够，会先 `evict_host`，仍不够则缩短预取长度或放弃。
- `ongoing_prefetch[req_id]` 记录 last_host_node、key、host_indices、operation，后续 progress/terminate 依赖这份账本。


## HiCacheController：后台线程与传输队列

Controller 把同步调度路径和慢 I/O 分开：

- write queue 合并 GPU->host 写入。
- prefetch queue / aux thread 处理 L3->host。
- ack queue 告诉 scheduler 哪些异步写已经完成。
- rate limit 避免 prefetch 占用过多 host pool。


In [ ]:
show_lines("python/sglang/srt/managers/cache_controller.py", 209, 305)
show_lines("python/sglang/srt/managers/cache_controller.py", 870, 910)
show_lines("python/sglang/srt/managers/cache_controller.py", 1024, 1070)


### `HiCacheController.write`：把多次写合并到异步 write stream

```python
# python/sglang/srt/managers/cache_controller.py:656-719
 656:     def write(
 657:         self,
 658:         device_indices: torch.Tensor,
 659:         priority: Optional[int] = None,
 660:         node_id: int = -1,
 661:     ) -> Optional[torch.Tensor]:
 662:         """
 663:         Back up KV caches from device memory to host memory.
 664:         """
 665:         host_indices = self.mem_pool_host.alloc(len(device_indices))
 666:         if host_indices is None:
 667:             return None
 668:         self.write_queue.append(
 669:             CacheOperation(host_indices, device_indices, node_id, priority)
 670:         )
 671:         self.start_writing()
 672:         return host_indices
 673: 
 674:     def start_writing(self) -> None:
 675:         if len(self.write_queue) == 0:
 676:             return
 677: 
 678:         op = CacheOperation.merge_ops(self.write_queue)
 679:         # Page-first write-back JIT kernels can keep destination host indices on CPU.
 680:         if (
 681:             self.io_backend == "kernel"
 682:             and self.mem_pool_host.layout == "page_first"
 683:             and getattr(self.mem_pool_host, "can_use_write_back_jit", False)
 684:         ):
 685:             host_indices, device_indices = op.host_indices, op.device_indices
 686:         else:
 687:             host_indices, device_indices = self.move_indices(
 688:                 op.host_indices, op.device_indices
 689:             )
 690:         self.write_queue.clear()
 691: 
 692:         start_event = device_module.Event()
 693:         finish_event = device_module.Event()
 694: 
 695:         start_event.record()
 696:         with device_module.stream(self.write_stream):
 697:             start_event.wait(self.write_stream)
 698:             self.mem_pool_host.backup_from_device_all_layer(
 699:                 self.mem_pool_device, host_indices, device_indices, self.io_backend
 700:             )
 701:             if self.has_draft:
 702:                 self.mem_pool_host_draft.backup_from_device_all_layer(
 703:                     self.mem_pool_device_draft,
 704:                     host_indices,
 705:                     device_indices,
 706:                     self.io_backend,
 707:                 )
 708:             finish_event.record()
 709:             # NOTE: We must save the host indices and device indices here,
 710:             # this is because we need to guarantee that these tensors are
 711:             # still alive when the write stream is executing.
 712:             if host_indices.is_cuda:
 713:                 host_indices.record_stream(self.write_stream)
 714:             if device_indices.is_cuda:
 715:                 device_indices.record_stream(self.write_stream)
 716: 
 717:         self.ack_write_queue.append(HiCacheAck(start_event, finish_event, op.node_ids))
 718: 
 719:     def load(
```

**读这段时抓住：**

- `write_queue` 先收集多个 cache operation，再 `merge_ops` 减少传输碎片。
- page-first layout 可能走 JIT write-back kernel，说明 host layout 是性能参数，不只是存储格式。
- `ack_write_queue` 用 start/finish event 把异步写完成状态交还给 scheduler。


### `prefetch_thread_func`：L3 命中数不足时会撤销预取

```python
# python/sglang/srt/managers/cache_controller.py:1024-1070
1024:     def prefetch_thread_func(self):
1025:         """
1026:         Manage prefetching operations from storage backend to host memory.
1027:         """
1028:         self.prefetch_buffer = Queue()
1029:         self.prefetch_io_aux_thread = threading.Thread(
1030:             target=self.prefetch_io_aux_func, daemon=True
1031:         )
1032:         self.prefetch_io_aux_thread.start()
1033:         while (not self.storage_stop_event.is_set()) or not self.prefetch_queue.empty():
1034:             try:
1035:                 operation = self.prefetch_queue.get(block=True, timeout=1)
1036:                 if operation is None:
1037:                     continue
1038:                 hash_value, storage_hit_count = self._storage_hit_query(operation)
1039:                 storage_hit_count_tensor = torch.tensor(
1040:                     storage_hit_count, dtype=torch.int
1041:                 )
1042:                 self._all_reduce_prefetch_groups(
1043:                     storage_hit_count_tensor, torch.distributed.ReduceOp.MIN
1044:                 )
1045:                 storage_hit_count = storage_hit_count_tensor.item()
1046: 
1047:                 if storage_hit_count < self.prefetch_threshold:
1048:                     # not to prefetch if not enough benefits
1049:                     self.prefetch_revoke_queue.put(operation.request_id)
1050:                     self.append_host_mem_release(operation.host_indices)
1051:                     logger.debug(
1052:                         f"Revoking prefetch for request {operation.request_id} due to insufficient hits ({storage_hit_count})."
1053:                     )
1054:                 else:
1055:                     operation.hash_value = hash_value[
1056:                         : (storage_hit_count // self.page_size)
1057:                     ]
1058:                     # free the pre-allocated memory for pages that are not hit
1059:                     self.append_host_mem_release(
1060:                         operation.host_indices[storage_hit_count:]
1061:                     )
1062:                     operation.host_indices = operation.host_indices[:storage_hit_count]
1063:                     logger.debug(
1064:                         f"Prefetching {len(operation.hash_value)} pages for request {operation.request_id}."
1065:                     )
1066:                     self.prefetch_buffer.put(operation)
1067: 
1068:             except Empty:
1069:                 continue
1070: 
```

**读这段时抓住：**

- storage prefetch 在后台线程里查询命中数量，并用 all-reduce 同步多 rank 结果。
- 命中 token 少于 `prefetch_threshold` 时不会真的发起 I/O，而是把 request id 放进 revoke queue。
- 这解释了 HiCache 不是“看到 L3 就读”，而是用阈值控制 latency/benefit tradeoff。


## Disaggregation：prefill/decode 分离为什么会影响 KV

PD disaggregation 把 prefill 和 decode 放到不同节点/进程。这样 prefill 产生的 KV 要跨节点交给 decode。HiCache 可以在 prefill/decode 节点上同时启用，用 L3 降低重复 prefill。

源码入口：

- `python/sglang/srt/disaggregation/prefill.py`
- `python/sglang/srt/disaggregation/decode.py`
- `python/sglang/srt/disaggregation/*/conn.py`
- `python/sglang/srt/disaggregation/kv_events.py`


In [ ]:
for path in [
    "python/sglang/srt/disaggregation/prefill.py",
    "python/sglang/srt/disaggregation/decode.py",
    "python/sglang/srt/disaggregation/kv_events.py",
]:
    print("\n==", path)
    for lineno, kind, name in find_defs(path):
        if "KV" in name or "Disagg" in name or name in {"PrefillBootstrapQueue", "DecodeScheduler"}:
            print(f"{lineno:4d} {kind:16s} {name}")


## 新特性常见落点

你看到 Advanced Features 新增页面时，可以先判断它属于哪类：

- **请求语义类**：采样参数、reasoning、tool parser、structured outputs。通常改 `io_struct.py`、OpenAI protocol、TokenizerManager、sampling。
- **调度/缓存类**：chunked prefill、Radix/HiCache、pause/resume、priority。通常改 Scheduler、ScheduleBatch、mem_cache。
- **执行优化类**：attention backend、CUDA graph、quantization、MoE。通常改 model_executor、layers、jit_kernel。
- **部署拓扑类**：PD/EPD disaggregation、gateway、multi-node。通常改 entrypoints、disaggregation、connector。
- **模型支持类**：新增 architecture。通常改 `srt/models`、weight loader、model registry、测试。


In [ ]:
advanced = sorted((ROOT / "docs" / "advanced_features").glob("*.md")) + sorted((ROOT / "docs" / "advanced_features").glob("*.ipynb"))
for p in advanced:
    print(p.name)


## 贡献者注意点

- HiCache 的正确性高度依赖多 rank 同步，任何“本 rank 看起来可以”的判断都要检查 all-reduce/barrier。
- L3 storage backend 不应泄漏模型执行细节；统一接口在 `HiCacheStorage`。
- 新 feature 如果改变 KV 生命周期，要同时考虑普通 RadixCache 和 HiRadixCache。
